# Session 4: media, annotations, and a dataset class

In session 3, you explored how a metadata table describes your collection. Part 1 returns briefly to shared fish data so everyone can practice turning related files into one usable sample and organizing loading behavior in a class. Part 2 is equally required: you will visualize several examples from your own data together with their labels or annotations so problems can be addressed before January.

# Part 1: shared loader and dataset-class pattern

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

In [ ]:
possible_data_dirs = [
    Path.cwd() / "shared_data" / "randalls_fish",
    Path.cwd().parent / "shared_data" / "randalls_fish",
    Path.cwd() / "Deliverables" / "prep" / "shared_data" / "randalls_fish",
]
data_dir = next((path for path in possible_data_dirs if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError(f"Could not find the supplied data. Current directory: {Path.cwd()}")
print("Data directory:", data_dir)

## 1. Match one image to its annotation

The image and CSV have the same **stem**: the filename without its suffix. Run this worked example and inspect both names.

In [ ]:
example_image_path = sorted(data_dir.glob("*.jpg"))[0]
example_annotation_path = example_image_path.with_suffix(".csv")
print("Image:     ", example_image_path.name)
print("Annotation:", example_annotation_path.name)
print("Same stem: ", example_image_path.stem == example_annotation_path.stem)

## 2. Build all valid pairs

**Your turn:** Complete `find_sample_pairs`. For every JPG, derive the CSV path with `with_suffix`. Raise `FileNotFoundError` if that CSV does not exist; otherwise append `(image_path, annotation_path)` to `pairs`.

In [ ]:
def find_sample_pairs(directory):
    pairs = []
    for image_path in sorted(Path(directory).glob("*.jpg")):
        # Derive and check annotation_path, then append the pair.
        pass
    return pairs

pairs = find_sample_pairs(data_dir)
print("Number of pairs:", len(pairs))
print("First pair:", pairs[0])

## 3. Inspect one annotation table

Before running the cell, predict what one row represents. Then inspect the column names and first rows. Which columns contain image coordinates?

**Prediction:** _Write here._

In [ ]:
annotation_table = pd.read_csv(example_annotation_path)
print("Columns:", annotation_table.columns.tolist())
display(annotation_table.head())

## 4. Write an annotation loader

**Your turn:** Complete `load_annotation`. Read the CSV, verify that it contains `X` and `Y`, and return those columns as a NumPy array. If columns are missing, raise a `ValueError` that names both the file and missing columns.

In [ ]:
def load_annotation(path):
    # Read, validate, and return the X/Y array.
    pass

example_points = load_annotation(example_annotation_path)
print("Point array shape:", example_points.shape)
print("First point:", example_points[0])

## 5. Complete `load_sample`

**Your turn:** Load the image as an RGB NumPy array, call `load_annotation`, and return the dictionary shown below. Keeping the paths and `sample_id` makes errors and plots easier to trace back to their source.

In [ ]:
def load_sample(image_path, annotation_path):
    # Load image and points here.
    return {
        "sample_id": image_path.stem,
        "image": None,       # Replace None.
        "points": None,      # Replace None.
        "image_path": image_path,
        "annotation_path": annotation_path,
    }

example_sample = load_sample(*pairs[0])
print("Sample ID:", example_sample["sample_id"])
print("Image shape:", example_sample["image"].shape)
print("Points shape:", example_sample["points"].shape)

## 6. Why use a class?

The functions work, but every caller must keep passing the directory and pair paths around. A dataset object stores that shared state once and gives the collection a consistent interface.

- `__init__` runs when the object is created and stores its state.
- `__len__` defines what `len(dataset)` returns.
- `__getitem__` defines what `dataset[index]` returns.

**Your turn:** Complete all three methods. Reuse the functions you already wrote; do not duplicate their loading logic.

In [ ]:
class FishDataset:
    def __init__(self, directory):
        # Store a Path and create the pairs list.
        pass

    def __len__(self):
        # Return the number of available pairs.
        pass

    def __getitem__(self, index):
        # Select a pair and return load_sample(...).
        pass

## 7. Use the dataset interface

Instantiate the class and run the tests. Then change the index to inspect another sample. In your own words, explain the difference between the `dataset` object and the dictionary returned by `dataset[0]`.

In [ ]:
dataset = FishDataset(data_dir)
print("Dataset length:", len(dataset))
sample = dataset[0]
print("Loaded sample:", sample["sample_id"])
print("Returned keys:", sorted(sample.keys()))

**Explanation:** _Write here._

## 8. Add a method

**Your turn:** Add a `describe(self)` method to the class above. It should print the data directory, number of samples, and first sample ID. Rerun the class cell, recreate `dataset`, and call `dataset.describe()` below.

In [ ]:
dataset = FishDataset(data_dir)
dataset.describe()

## 9. Diagnose a missing annotation

The next cell creates a normal dataset, then deliberately replaces the annotation path in its first stored pair with a nonexistent path. Predict whether the error will arise on the assignment, in `__len__`, or in `__getitem__`. Run the cell, read the error, and explain why. This changes only the in-memory list; it does not change any files.

**Prediction and explanation:** _Write here._

In [ ]:
broken_dataset = FishDataset(data_dir)
image_path, annotation_path = broken_dataset.pairs[0]
broken_dataset.pairs[0] = (image_path, data_dir / "missing_annotation.csv")
print("Dataset still has", len(broken_dataset), "items")
broken_dataset[0]

## 10. Plot a sample returned by the class

**Your turn:** Display `sample["image"]`, then overlay `sample["points"]` with `ax.scatter`. The X coordinates are column 0 and Y coordinates are column 1. Use the sample ID as the title.

In [ ]:
sample = dataset[0]
fig, ax = plt.subplots(figsize=(8, 5))
# Display the image and overlay the points here.
ax.axis("off")
plt.show()

# Part 2: visualize your own media and labels

This is important. A metadata summary cannot reveal shifted boxes, incorrect labels, empty masks, coordinate-system mistakes, corrupt media, or annotations linked to the wrong sample. Choose one pathway below and adapt the provided code with a coding agent (or the help of your trusty instructors). You do **not** need to run all pathways, choose the one that is appropriate to your project.

Use a small local sample and keep research data outside this repository. Inspect several examples representing the variation in the data (e.g., different sites, species, behavior classes, etc). Include typical examples and, when possible, difficult or questionable examples identified during session 3.

## Configure your pathway

Choose `classification`, `detection`, `keypoints`, `segmentation`, `geospatial`, `audio`, or `video`. Most image pathways use a CSV and media directory. Column names and annotation formats vary, so changing the settings is expected.

In [ ]:
PROJECT_TASK = None  # Example: "classification"
OWN_METADATA_PATH = None  # Path to a small CSV extract
OWN_MEDIA_DIR = None  # Directory containing the corresponding media
FILENAME_COLUMN = "filename"
LABEL_COLUMN = "label"

if OWN_METADATA_PATH is not None:
    own_metadata = pd.read_csv(Path(OWN_METADATA_PATH).expanduser())
    print("Loaded metadata rows:", len(own_metadata))
    display(own_metadata.head())
else:
    own_metadata = None
    print("Set PROJECT_TASK and your paths with an instructor before using a pathway.")

## Pathway A: classification grid

Use this when each image has one class label. Select a manageable set of classes and show multiple randomly selected images from each class. A grid makes within-class variation and obvious label errors visible.

In [ ]:
CLASSES_TO_SHOW = None  # Example: ["deer", "fox", "empty"]
EXAMPLES_PER_CLASS = 3

if PROJECT_TASK == "classification":
    media_dir = Path(OWN_MEDIA_DIR).expanduser()
    classes = CLASSES_TO_SHOW or own_metadata[LABEL_COLUMN].dropna().unique()[:4]
    fig, axes = plt.subplots(len(classes), EXAMPLES_PER_CLASS, figsize=(4 * EXAMPLES_PER_CLASS, 3.5 * len(classes)), squeeze=False)
    for row_index, class_name in enumerate(classes):
        candidates = own_metadata[own_metadata[LABEL_COLUMN] == class_name]
        chosen = candidates.sample(min(EXAMPLES_PER_CLASS, len(candidates)), random_state=7)
        for column_index, ax in enumerate(axes[row_index]):
            ax.axis("off")
            if column_index < len(chosen):
                record = chosen.iloc[column_index]
                image_path = media_dir / str(record[FILENAME_COLUMN])
                with Image.open(image_path) as opened_image:
                    ax.imshow(opened_image.convert("RGB"))
                ax.set_title(f"{class_name}\n{image_path.name}")
    plt.tight_layout()
    plt.show()

## Pathway B: bounding boxes

Use this when the annotation table has one row per box. This template assumes pixel coordinates `(xmin, ymin, xmax, ymax)`; convert first if your format stores normalized coordinates, center/width/height, or another convention.

In [ ]:
BOX_COLUMNS = {"xmin": "xmin", "ymin": "ymin", "xmax": "xmax", "ymax": "ymax"}
DETECTION_FILES_TO_SHOW = None

if PROJECT_TASK == "detection":
    from matplotlib.patches import Rectangle
    media_dir = Path(OWN_MEDIA_DIR).expanduser()
    filenames = DETECTION_FILES_TO_SHOW or own_metadata[FILENAME_COLUMN].dropna().unique()[:4]
    fig, axes = plt.subplots(1, len(filenames), figsize=(5 * len(filenames), 5), squeeze=False)
    for ax, filename in zip(axes[0], filenames):
        image_path = media_dir / str(filename)
        with Image.open(image_path) as opened_image:
            ax.imshow(opened_image.convert("RGB"))
        boxes = own_metadata[own_metadata[FILENAME_COLUMN] == filename]
        for _, box in boxes.iterrows():
            xmin, ymin = box[BOX_COLUMNS["xmin"]], box[BOX_COLUMNS["ymin"]]
            xmax, ymax = box[BOX_COLUMNS["xmax"]], box[BOX_COLUMNS["ymax"]]
            ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color="red", linewidth=2))
            ax.text(xmin, ymin, str(box[LABEL_COLUMN]), color="white", backgroundcolor="red")
        ax.set_title(str(filename))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Pathway C: keypoints or landmarks

Use this when the annotation table has one row per point. If points are normalized rather than pixels, convert them using the image width and height before plotting.

In [ ]:
X_COLUMN, Y_COLUMN = "X", "Y"
KEYPOINT_FILES_TO_SHOW = None

if PROJECT_TASK == "keypoints":
    media_dir = Path(OWN_MEDIA_DIR).expanduser()
    filenames = KEYPOINT_FILES_TO_SHOW or own_metadata[FILENAME_COLUMN].dropna().unique()[:4]
    fig, axes = plt.subplots(1, len(filenames), figsize=(5 * len(filenames), 5), squeeze=False)
    for ax, filename in zip(axes[0], filenames):
        image_path = media_dir / str(filename)
        with Image.open(image_path) as opened_image:
            ax.imshow(opened_image.convert("RGB"))
        points = own_metadata[own_metadata[FILENAME_COLUMN] == filename]
        ax.scatter(points[X_COLUMN], points[Y_COLUMN], c="red", s=20)
        ax.set_title(str(filename))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Pathway D: segmentation masks

Use this when each image has a corresponding mask image. The overlay should line up with object boundaries; mismatched dimensions are an immediate problem to record.

In [ ]:
MASK_FILENAME_COLUMN = "mask_filename"
OWN_MASK_DIR = None
SEGMENTATION_ROWS_TO_SHOW = None

if PROJECT_TASK == "segmentation":
    media_dir = Path(OWN_MEDIA_DIR).expanduser()
    mask_dir = Path(OWN_MASK_DIR).expanduser()
    row_indices = SEGMENTATION_ROWS_TO_SHOW or own_metadata.index[:4].tolist()
    fig, axes = plt.subplots(1, len(row_indices), figsize=(5 * len(row_indices), 5), squeeze=False)
    for ax, row_index in zip(axes[0], row_indices):
        record = own_metadata.loc[row_index]
        with Image.open(media_dir / str(record[FILENAME_COLUMN])) as opened_image:
            image = opened_image.convert("RGB")
        with Image.open(mask_dir / str(record[MASK_FILENAME_COLUMN])) as opened_mask:
            mask = np.asarray(opened_mask)
        if image.size != (mask.shape[1], mask.shape[0]):
            raise ValueError(f"Image and mask sizes differ for row {row_index}: {image.size} versus {mask.shape}")
        ax.imshow(image)
        ax.imshow(mask, alpha=0.4, cmap="magma")
        ax.set_title(str(record[FILENAME_COLUMN]))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Pathway E: geospatial tiles and annotations

If a raster tile and its labels already use pixel coordinates, use the detection, keypoint, or segmentation pathway above. If annotations are geographic vector data, use `rasterio` and `geopandas` to read both sources, transform the vectors to the raster's CRS, and overlay them. These packages are not part of the common prep environment, so do this with an instructor in the environment already used for your geospatial work.

The critical checks are: the raster and vectors report compatible coordinate reference systems; the vectors appear over the intended tile rather than far away; and annotation geometry aligns with visible features.

In [ ]:
RASTER_TILE_PATH = None
VECTOR_ANNOTATION_PATH = None

if PROJECT_TASK == "geospatial":
    import geopandas as gpd
    import rasterio
    from rasterio.plot import show
    with rasterio.open(Path(RASTER_TILE_PATH).expanduser()) as raster:
        vectors = gpd.read_file(Path(VECTOR_ANNOTATION_PATH).expanduser()).to_crs(raster.crs)
        print("Raster CRS:", raster.crs)
        print("Annotation CRS after conversion:", vectors.crs)
        fig, ax = plt.subplots(figsize=(8, 8))
        show(raster, ax=ax)
        vectors.boundary.plot(ax=ax, color="red", linewidth=2)
        ax.set_title(Path(RASTER_TILE_PATH).name)
        plt.show()

## Audio and video projects

The same requirement applies, but a still-image grid may not be the right view. For audio, plot several waveforms or spectrograms and overlay labeled time intervals. For video, display representative frames and overlay frame-level labels, boxes, points, or masks. Use `modality_examples.py` from session 2 as the loading starting point and adapt the label overlay with an instructor.

## Record what you found

Make a few slides to show next group meeting, include sources of variation in your dataset, any correlations that might affect the performance of your system and any other issues you can think of and any cool examples you find while exploring.
